# linear-affine-on-custom-tensor — ex1: forward pass of Linear over hand-written Tensor wrappers

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linear-affine-on-custom-tensor`. Running the final beacon cell reports progress against the `Backprop: Linear affine on custom Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Linear affine on custom Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linear-affine-on-custom-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linear-affine-on-custom-tensor"
DD_SUBTOPIC = "Backprop: Linear affine on custom Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Linear affine on custom Tensor — quick refresher

A `Linear` layer is just the affine map `out = input @ weight + bias`. When you build it on YOUR hand-written `Tensor` (rather than `torch.nn.Linear`), TWO things change:

1. **`@` and `+` must be wrapped ops.** Each produces a new `MiniTensor` with a `Recipe`, so the reverse pass can find the parents (`input`, `weight`, `bias`) and call the right `_back` fns.
2. **`weight` and `bias` are `Parameter` (Tensor subclass).** They're leaves with `requires_grad=True`, so the wrapped `@` / `+` propagate grad through them via the OR rule.

```python
class Linear:
    def __init__(self, in_f, out_f):
        self.weight = Parameter(t.empty(in_f, out_f))   # init separately
        self.bias   = Parameter(t.zeros(out_f))
    def forward(self, x: MiniTensor) -> MiniTensor:
        return mm(x, self.weight) + self.bias            # broadcast over batch
```

Output shape: `(B, in_f) @ (in_f, out_f) + (out_f,)` → `(B, out_f)`. The bias broadcasts over the batch axis — bias gradient is the broadcast-sum.

### Exercise 1 — forward pass of Linear over hand-written Tensor wrappers

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `out = input @ weight + bias` affine map over the custom MiniTensor wrappers, producing a result MiniTensor whose Recipe names matmul as its forward fn and lists both Parameter inputs as parents.
> Keywords: linear, affine, matmul, bias-broadcast, custom-tensor
> ```

**KCs targeted:** `linear-affine-on-custom-tensor`, `box-array-to-tensor-with-recipe`

Implement `linear_forward(x, weight, bias)` — the forward pass of an ARENA-style `Linear` layer over MiniTensors. The drill is about wiring the affine map correctly on YOUR wrapper class; the autograd you'd want underneath is mocked by a simple Recipe attached to the matmul output.

Inputs:
- `x`:      `MiniTensor` of shape `(B, in_features)` — the batched input.
- `weight`: `MiniTensor` of shape `(in_features, out_features)` — a Parameter.
- `bias`:   `MiniTensor` of shape `(out_features,)` — a Parameter.

Behavior:
1. Compute `mm_arr = x.array @ weight.array`, shape `(B, out_features)`.
2. Wrap as `mm = MiniTensor(mm_arr, requires_grad=(x.requires_grad or weight.requires_grad))` and attach `mm.recipe = Recipe(func=t.matmul, args=(x.array, weight.array), kwargs={}, parents={0: x, 1: weight})`.
3. Compute `out_arr = mm.array + bias.array` (bias broadcasts over the batch axis).
4. Wrap as `out = MiniTensor(out_arr, requires_grad=(mm.requires_grad or bias.requires_grad))` and attach `out.recipe = Recipe(func=t.add, args=(mm.array, bias.array), kwargs={}, parents={0: mm, 1: bias})`.
5. Return `out`.

Why the two-level Recipe. A real ARENA pipeline replaces the Recipe-construction lines with `mm = wrapped_matmul(x, weight)` and `out = wrapped_add(mm, bias)` — `wrap_forward_fn` builds the same Recipe. We construct manually here so the drill stays focused on the affine-map mechanics.

In [ ]:
def linear_forward(x: MiniTensor, weight: MiniTensor, bias: MiniTensor) -> MiniTensor:
    # step 1: matmul
    mm_arr = x.array @ weight.array
    mm = MiniTensor(
        mm_arr,
        requires_grad=(x.requires_grad or weight.requires_grad),
    )
    mm.recipe = Recipe(
        func=t.matmul,
        args=(x.array, weight.array),
        kwargs={},
        parents={0: x, 1: weight},
    )
    # step 2: bias add (broadcasts over batch axis)
    out_arr = mm.array + bias.array
    out = MiniTensor(
        out_arr,
        requires_grad=(mm.requires_grad or bias.requires_grad),
    )
    out.recipe = Recipe(
        func=t.add,
        args=(mm.array, bias.array),
        kwargs={},
        parents={0: mm, 1: bias},
    )
    return out


<details><summary>Solution</summary>

```python
def linear_forward(x: MiniTensor, weight: MiniTensor, bias: MiniTensor) -> MiniTensor:
    # step 1: matmul
    mm_arr = x.array @ weight.array
    mm = MiniTensor(
        mm_arr,
        requires_grad=(x.requires_grad or weight.requires_grad),
    )
    mm.recipe = Recipe(
        func=t.matmul,
        args=(x.array, weight.array),
        kwargs={},
        parents={0: x, 1: weight},
    )
    # step 2: bias add (broadcasts over batch axis)
    out_arr = mm.array + bias.array
    out = MiniTensor(
        out_arr,
        requires_grad=(mm.requires_grad or bias.requires_grad),
    )
    out.recipe = Recipe(
        func=t.add,
        args=(mm.array, bias.array),
        kwargs={},
        parents={0: mm, 1: bias},
    )
    return out
```

**Two ops, two Recipes.** The Recipe chain is `x -> mm -> out`. The intermediate `mm` is its own MiniTensor (so the reverse pass can find weight as `mm.recipe.parents[1]`). Collapsing into one Recipe would conflate matmul-back and add-back into a single dispatch — exactly what `wrap_forward_fn` avoids.

**Bias broadcasts in the forward.** Shape `(B, out_f) + (out_f,)` uses the standard right-aligned broadcast rules. The corresponding `add_back1` for the bias path has to `unbroadcast` over the batch axis — sum out the leading `B` dim — to recover the `(out_f,)` grad shape. That's why the bias-grad code is `grad_out.sum(dim=0)`.

**Why this is the capstone shape.** A working `linear_forward` plus matmul-back + add-back gives you a fully end-to-end-trainable Linear layer on your hand-built autograd. From here, stacking layers, ReLU, cross-entropy, and SGD assembles a complete MNIST training loop without touching `torch.autograd` once.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()